# 14 — Negativos difíciles: criterios para el muestreo del cargador

El cargador entrena con todos los positivos y, en cada época, con una muestra de los negativos tomada de dos bolsas: **difíciles** y **aleatorios** (proporción inicial 1 : 2 : 2, parametrizable; ver `claude/plan_modelado.md`, S3). Esta notebook calcula los atributos que deciden a qué bolsa pertenece cada negativo. No modifica las etiquetas de `05-ground_truth_labels.ipynb` ni el índice de `11-dataset_training.ipynb`: escribe tablas auxiliares que el cargador cruza con el índice.

Criterios de negativo difícil (se activan por parámetro en el cargador; qué combinación conviene se compara en S7):

| Criterio | Definición | Estado |
|-|-|-|
| `red` | en el mismo t, otra estación de la red es positiva | en el índice (`n_est_pos`), a recalcular sin Nueva Generación |
| `red_activa` | además cuentan las estaciones con lluvia `en_curso` o tormenta `inminente` en el mismo t | por calcular |
| `casi_positivo` | misma estación, intensidad horaria máxima en la ventana entre 50 % y 80 % del umbral | **Sección 1** |
| `temporal` | misma estación con un positivo a ≤ 24 h | en el índice (`horas_a_positivo`) |
| `nube_fria` | p10 de C13 del parche en t frío | por calcular (desde el caché) |

Nueva Generación (`0021206600`) se excluye del modelado (decisión del 8-oct-2026): no aparece en ningún conjunto ni cuenta como estación positiva o activa para definir negativos difíciles.

In [1]:
import sys, pathlib
_raiz = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / ".gitignore").exists())
sys.path.insert(0, str(_raiz))
import entorno

In [2]:
# Configuración --------------------------------------------------------------------
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

RUTA_BASE   = Path(r"C:\Users\diani\Documents\repositories\nowcasting-bogota")
RUTA_GT     = RUTA_BASE / "data_ingestion" / "ground_truth"
RUTA_CSV    = RUTA_BASE / "data" / "rainfall_subhourly"
RUTA_DS     = RUTA_BASE / "datasets"
RUTA_INT    = RUTA_DS / "intensidad_ventana.parquet"        # salida de la Sección 1 (git-ignorada)

sys.path.append(str(RUTA_BASE / "src"))
from negativos import intensidad_ventana_estacion, banda_intensidad, BANDAS_INT

NG = "0021206600"                                           # Nueva Generación: fuera del modelado
ORDEN = ["train", "val", "test"]

def cod10(s: pd.Series) -> pd.Series:
    # Código de estación como texto de 10 dígitos (las tablas lo guardan como texto o como entero)
    return s.astype("int64").astype(str).str.zfill(10)

umbrales  = pd.read_csv(RUTA_GT / "umbral_operativo_por_estacion.csv", dtype={"codigoestacion": str}).set_index("codigoestacion")
etiquetas = pd.read_parquet(RUTA_GT / "etiquetas_ground_truth.parquet")      # 05
indice    = pd.read_parquet(RUTA_DS / "indice_completo.parquet")             # 11
csvs      = {p.stem: p for p in RUTA_CSV.glob("*.csv")}

etiquetas["codigoestacion"] = cod10(etiquetas["codigoestacion"])
indice["codigoestacion"]    = cod10(indice["codigoestacion"])

assert len(umbrales) == 63 and NG in umbrales.index and not (set(umbrales.index) - set(csvs))
for nombre, d in [("etiquetas", etiquetas), ("indice", indice)]:
    print(f"{nombre:9s}: {len(d):>10,} filas | timestamp {d['timestamp'].dtype}, tz = {d['timestamp'].dt.tz}")
assert str(etiquetas["timestamp"].dt.tz) == str(indice["timestamp"].dt.tz) == "UTC"

etiquetas:  6,200,666 filas | timestamp datetime64[ns, UTC], tz = UTC
indice   :  4,029,925 filas | timestamp datetime64[ns, UTC], tz = UTC


## 1. Intensidad máxima en la ventana (`casi_positivo`)

Para cada par (estación, t) se calcula `int_max_ventana`: el máximo de la intensidad rolling 60 min en τ ∈ (t+2h, t+3h], es decir, sobre las horas completas contenidas en la ventana (t+1h, t+3h]. Es exactamente la cantidad con que se define la etiqueta: un par es positivo si y solo si `int_max_ventana ≥ umbral`. La función (`src/negativos.py`) reutiliza `bins_estacion` de `src/etiquetado.py`, de modo que la grilla, los bins y el tratamiento de los bins sin registros son los mismos que en 05.

`int_t` (ya presente en las etiquetas) es la intensidad en t, no en la ventana, por eso hace falta este cálculo.

In [3]:
resultados = []
for cod in tqdm(sorted(umbrales.index), desc="Intensidad en la ventana"):
    d = intensidad_ventana_estacion(csvs[cod])
    d.insert(0, "codigoestacion", cod)
    resultados.append(d)

intensidad = pd.concat(resultados, ignore_index=True)
assert not intensidad.duplicated(["codigoestacion", "timestamp"]).any()
print(f"Pares: {len(intensidad):,} | tz: {intensidad['timestamp'].dt.tz}")

Intensidad en la ventana: 100%|██████████| 63/63 [00:40<00:00,  1.54it/s]


Pares: 6,200,666 | tz: UTC


### 1.1 Verificación contra las etiquetas de 05

La condición es exacta, sin tolerancia: en todos los pares, `etiqueta == 1` si y solo si `int_max_ventana ≥ umbral`. Los pares con etiqueta NaN (sin cobertura suficiente) deben quedar bajo el umbral, porque la regla de cobertura asimétrica declara positivo todo par que supere el umbral aunque falten registros.

In [4]:
v = etiquetas[["codigoestacion", "timestamp", "etiqueta"]].merge(
        intensidad, on=["codigoestacion", "timestamp"], how="outer", indicator=True)
print(v["_merge"].value_counts().to_string())
assert (v["_merge"] == "both").all(), "Las grillas de etiquetas e intensidad no coinciden"

v["umbral"] = v["codigoestacion"].map(umbrales["umbral_mm_h"])
supera = v["int_max_ventana"] >= v["umbral"]
pos = v["etiqueta"] == 1
print(f"\nPositivos: {int(pos.sum()):,} | pares que superan el umbral: {int(supera.sum()):,}")
print(pd.crosstab(v["etiqueta"].fillna(-1).map({1: "1", 0: "0", -1: "NaN"}), supera.map({True: "≥ umbral", False: "< umbral"}),
                  rownames=["etiqueta"], colnames=["int_max_ventana"]).to_string())
assert (supera == pos).all(), "La intensidad en la ventana no reproduce la etiqueta"
print("\nVerificación exacta: etiqueta == 1 ⇔ int_max_ventana ≥ umbral")

_merge
both          6200666
left_only           0
right_only          0

Positivos: 18,942 | pares que superan el umbral: 18,942
int_max_ventana  < umbral  ≥ umbral
etiqueta                           
0                 4064291         0
1                       0     18942
NaN               2117433         0

Verificación exacta: etiqueta == 1 ⇔ int_max_ventana ≥ umbral


In [5]:
intensidad.to_parquet(RUTA_INT, index=False)
print(f"Guardado: {RUTA_INT.name} | {len(intensidad):,} pares | columnas: {list(intensidad.columns)}")

Guardado: intensidad_ventana.parquet | 6,200,666 pares | columnas: ['codigoestacion', 'timestamp', 'int_max_ventana']


### 1.2 Cuántos negativos aporta cada banda de intensidad

Se cruza el índice de 11 (pares con historia L_max ≥ 2, sin `en_curso` ni `inminente`) con la intensidad en la ventana, sin Nueva Generación. La razón `int_max_ventana / umbral` se agrupa en bandas; el criterio `casi_positivo` es la banda 50–80 %. La banda 80–100 % no se muestrea como difícil (cae dentro de la incertidumbre del P95, CV mediano 18.7 %) y se reserva para el análisis de errores en S12.

In [6]:
ind = indice[indice["codigoestacion"] != NG].merge(intensidad, on=["codigoestacion", "timestamp"], how="left")
assert ind["int_max_ventana"].notna().all(), "Pares del índice sin intensidad en la ventana"
ind["razon"] = ind["int_max_ventana"] / ind["codigoestacion"].map(umbrales["umbral_mm_h"])
ind["banda_int"] = banda_intensidad(ind["razon"])

# Coherencia: positivos siempre ≥ umbral, negativos siempre por debajo
assert (ind.loc[ind["etiqueta"] == 1, "banda_int"] == "≥ umbral").all()
assert (ind.loc[ind["etiqueta"] == 0, "banda_int"] != "≥ umbral").all()
print(f"Índice sin Nueva Generación: {len(ind):,} pares (se quitan {len(indice) - len(ind):,})")

filas = []
for p in ORDEN:
    for L in ((2, 4) if p == "train" else (4,)):
        d = ind[(ind["periodo"] == p) & (ind["L_max"] >= L)]
        n_pos = int((d["etiqueta"] == 1).sum())
        cnt = d.loc[d["etiqueta"] == 0, "banda_int"].value_counts().reindex(BANDAS_INT[:4])
        filas.append({"periodo": p, "uso": f"L_max ≥ {L}", "positivos": n_pos,
                      **{b: int(n) for b, n in cnt.items()},
                      "50–80 % por positivo": round(cnt["50–80 %"] / n_pos, 1)})
tabla_bandas = pd.DataFrame(filas)
print("\nNegativos por banda de intensidad en la ventana:")
print(tabla_bandas.to_string(index=False))

Índice sin Nueva Generación: 3,973,266 pares (se quitan 56,659)

Negativos por banda de intensidad en la ventana:
periodo       uso  positivos  sin lluvia  < 50 %  50–80 %  80–100 %  50–80 % por positivo
  train L_max ≥ 2       4249     1143548   90621     4302      1437                   1.0
  train L_max ≥ 4       4155     1138859   90153     4249      1411                   1.0
    val L_max ≥ 4       1487      299620   22143     1606       569                   1.1
   test L_max ≥ 4      12320     2062437  303030    13897      3806                   1.1


### 1.3 Solapamiento con el criterio espacial vigente

Cuántos negativos de entrenamiento casi positivos ya eran near-miss por la regla de 11 (`pool == "near_miss"`, que todavía incluye a Nueva Generación entre las positivas; se recalcula en la Sección 2). Si el solapamiento es alto, el criterio nuevo aporta poco; si es bajo, amplía la bolsa difícil con escenas que la regla espacial no veía.

In [7]:
neg_tr = ind[(ind["periodo"] == "train") & (ind["etiqueta"] == 0)]
casi = neg_tr["banda_int"] == "50–80 %"
print(pd.crosstab(neg_tr["pool"], casi.map({True: "casi_positivo", False: "otro"}), margins=True).to_string())

banda_int  casi_positivo     otro      All
pool                                      
aleatorio           1087  1182572  1183659
near_miss           3215    53034    56249
All                 4302  1235606  1239908


### Conclusión de la Sección 1

La intensidad horaria máxima en la ventana se calculó para los 6,200,666 pares de las etiquetas y reproduce la etiqueta sin excepción: los 18,942 positivos superan el umbral y ninguno de los 4,064,291 negativos ni de los 2,117,433 pares sin cobertura lo hace. El resultado quedó en `intensidad_ventana.parquet`.

Sin Nueva Generación, el índice conserva 3,973,266 pares. Los negativos casi positivos (50–80 % del umbral) son escasos, alrededor de uno por positivo en todos los períodos (4,302 en train con L_max ≥ 2, 1,606 en val y 13,897 en test), y en su mayoría ya eran near-miss espaciales (3,215 de 4,302 en train, 75 %). El criterio agrega 1,087 negativos que antes caían en la bolsa aleatoria. Esto es coherente con la física del problema: una estación que recibe una fracción alta de su umbral suele estar bajo una tormenta que lo supera en otra estación de la red. El criterio queda disponible en el cargador, pero se espera un efecto pequeño. La banda 80–100 % (1,437 negativos en train y 3,806 en test) no se muestrea y se reserva para el análisis de errores.

## 2. Criterios espaciales sin Nueva Generación: `red` y `red_activa`

**2.1 `red` recalculado.** En 11, `n_est_pos` y `dist_pos_km` se calcularon con todos los positivos utilizables de 05, incluida Nueva Generación. Como NG sale del modelado, un negativo cuya única "vecina positiva" era NG deja de ser near-miss. Primero se comprueba que el cálculo reproduce exactamente el `n_est_pos` de 11 con NG; luego se recalcula sin ella.

**2.2 `red_activa`.** Una estación está *activa* en t si es positiva (como en `red`), si tiene lluvia fuerte `en_curso` en t o si tiene una tormenta `inminente` (lluvia fuerte que empieza y termina antes de la ventana). Esos pares se excluyeron del conjunto en 11, por eso un negativo vecino a una tormenta ya en curso quedaba como aleatorio. `n_est_activas` cuenta las otras estaciones activas en el mismo t, sin NG. Las columnas `en_curso` e `inminente` se leen de las etiquetas de 05, en UTC con zona como el índice.

In [8]:
# 2.1 Control: con NG se reproduce el n_est_pos de 11 ------------------------------
utiles = etiquetas["etiqueta"].notna() & ~etiquetas["en_curso"] & ~etiquetas["inminente"]   # pares de 11
es_pos = utiles & (etiquetas["etiqueta"] == 1)
cnt_con_ng = etiquetas.loc[es_pos].groupby("timestamp").size()
rep = indice["timestamp"].map(cnt_con_ng).fillna(0).astype(int) - (indice["etiqueta"] == 1).astype(int)
assert (rep.to_numpy() == indice["n_est_pos"].to_numpy()).all(), "No se reproduce n_est_pos de 11"
print("Control: el conteo con NG reproduce exactamente n_est_pos de 11")

# Recalculo sin NG
sin_ng = etiquetas["codigoestacion"] != NG
pos_ref = etiquetas.loc[es_pos & sin_ng, ["codigoestacion", "timestamp"]]
ind = ind.rename(columns={"n_est_pos": "n_est_pos_11", "dist_pos_km": "dist_pos_km_11"})
ind["n_est_pos"] = (ind["timestamp"].map(pos_ref.groupby("timestamp").size()).fillna(0).astype(int)
                    - (ind["etiqueta"] == 1).astype(int))
assert (ind["n_est_pos"] <= ind["n_est_pos_11"]).all()

# dist_pos_km sin NG: misma construcción que 11 (distancia entre píxeles × 2 km)
KM_PX = 2.0
pixels = pd.read_csv(RUTA_BASE / "data_ingestion" / "satellite" / "estaciones_pixels.csv")
px = pixels.assign(codigoestacion=cod10(pixels["codigoestacion"])).set_index("codigoestacion")
P  = px[["fila", "col"]].to_numpy(float)
D  = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).sum(-1)) * KM_PX
est_idx = {c: i for i, c in enumerate(px.index)}

pos_por_t = pos_ref.groupby("timestamp")["codigoestacion"].apply(lambda s: np.array([est_idx[c] for c in s]))
dist = np.full(len(ind), np.nan)
sub = ind[ind["timestamp"].isin(pos_por_t.index)]
for t, gt in tqdm(sub.groupby("timestamp"), desc="dist_pos_km"):
    pe = pos_por_t[t]
    ei = gt["codigoestacion"].map(est_idx).to_numpy()
    Dm = D[np.ix_(ei, pe)].copy()
    Dm[ei[:, None] == pe[None, :]] = np.inf
    d = Dm.min(axis=1)
    dist[gt.index] = np.where(np.isinf(d), np.nan, d)
ind["dist_pos_km"] = dist

neg = ind["etiqueta"] == 0
cambio = neg & (ind["n_est_pos_11"] >= 1) & (ind["n_est_pos"] == 0)
print("\nNegativos que dejan de ser near-miss al quitar NG (su única vecina positiva era NG):")
print(ind[cambio].groupby("periodo").size().reindex(ORDEN, fill_value=0).to_string())

Control: el conteo con NG reproduce exactamente n_est_pos de 11


dist_pos_km: 100%|██████████| 5642/5642 [00:03<00:00, 1852.63it/s]


Negativos que dejan de ser near-miss al quitar NG (su única vecina positiva era NG):
periodo
train     338
val      1945
test     1946


In [9]:
# 2.2 red_activa ---------------------------------------------------------------------
activa = es_pos | etiquetas["en_curso"] | etiquetas["inminente"]
act = etiquetas.loc[activa & sin_ng]
print("Estaciones activas por motivo (pares estación-t, sin NG):")
print(pd.Series({"positiva": int((es_pos & sin_ng).sum()),
                 "en_curso": int((etiquetas["en_curso"] & sin_ng).sum()),
                 "inminente": int((etiquetas["inminente"] & sin_ng).sum())}).to_string())

ind["n_est_activas"] = (ind["timestamp"].map(act.groupby("timestamp").size()).fillna(0).astype(int)
                        - (ind["etiqueta"] == 1).astype(int))      # el propio par nunca es en_curso/inminente
assert (ind["n_est_activas"] >= ind["n_est_pos"]).all()

neg_tr = ind[(ind["periodo"] == "train") & (ind["etiqueta"] == 0)]
print("\nNegativos de train (L_max ≥ 2): red frente a red_activa")
print(pd.crosstab(neg_tr["n_est_pos"] >= 1, neg_tr["n_est_activas"] >= 1,
                  rownames=["red"], colnames=["red_activa"], margins=True).to_string())

Estaciones activas por motivo (pares estación-t, sin NG):
positiva     18475
en_curso     10345
inminente    11073

Negativos de train (L_max ≥ 2): red frente a red_activa
red_activa    False   True      All
red                                
False       1144266  39731  1183997
True              0  55911    55911
All         1144266  95642  1239908


### 2.3 Tamaño de la bolsa difícil por combinación de criterios

Cada fila es una combinación posible de criterios (unión). Se reporta el número de negativos de train que quedan en la bolsa difícil y cuántos hay por positivo: con la proporción 1 : 2 : 2 se necesitan 2 por positivo en cada época, de modo que una bolsa de 13 por positivo se recorre completa en unas 7 épocas. `nube_fria` se agrega en la Sección 3.

In [10]:
crit = {
    "red":           ind["n_est_pos"] >= 1,
    "red_activa":    ind["n_est_activas"] >= 1,
    "casi_positivo": ind["banda_int"] == "50–80 %",
    "temporal":      ind["horas_a_positivo"] <= 24,
}
combos = [["red"], ["red_activa"], ["red", "casi_positivo"], ["red_activa", "casi_positivo"],
          ["red_activa", "casi_positivo", "temporal"]]

filas = []
for p in ORDEN:
    for L in ((2, 4) if p == "train" else (4,)):
        m = (ind["periodo"] == p) & (ind["L_max"] >= L)
        n_pos = int((m & (ind["etiqueta"] == 1)).sum())
        n_neg = int((m & (ind["etiqueta"] == 0)).sum())
        for c in combos:
            dif = m & (ind["etiqueta"] == 0) & np.logical_or.reduce([crit[k] for k in c])
            filas.append({"periodo": p, "uso": f"L_max ≥ {L}", "criterios": " ∪ ".join(c),
                          "difíciles": int(dif.sum()), "por positivo": round(dif.sum() / n_pos, 1),
                          "% de negativos": f"{dif.sum() / n_neg:.1%}"})
bolsas = pd.DataFrame(filas)
print(bolsas.to_string(index=False))

periodo       uso                             criterios  difíciles  por positivo % de negativos
  train L_max ≥ 2                                   red      55911          13.2           4.5%
  train L_max ≥ 2                            red_activa      95642          22.5           7.7%
  train L_max ≥ 2                   red ∪ casi_positivo      56998          13.4           4.6%
  train L_max ≥ 2            red_activa ∪ casi_positivo      96333          22.7           7.8%
  train L_max ≥ 2 red_activa ∪ casi_positivo ∪ temporal     161572          38.0          13.0%
  train L_max ≥ 4                                   red      55235          13.3           4.5%
  train L_max ≥ 4                            red_activa      94555          22.8           7.7%
  train L_max ≥ 4                   red ∪ casi_positivo      56318          13.6           4.6%
  train L_max ≥ 4            red_activa ∪ casi_positivo      95242          22.9           7.7%
  train L_max ≥ 4 red_activa ∪ casi_posi

### 2.4 Tabla de atributos para el cargador

Una fila por par del índice sin Nueva Generación, con las columnas que el cargador cruza para formar las bolsas. Se guarda aparte del índice de 11, que no se modifica.

In [11]:
RUTA_ATR = RUTA_DS / "negativos_atributos.parquet"
atributos = ind[["codigoestacion", "timestamp", "n_est_pos", "dist_pos_km", "n_est_activas",
                 "horas_a_positivo", "int_max_ventana", "razon"]].copy()
assert len(atributos) == len(indice) - int((indice["codigoestacion"] == NG).sum())
assert not atributos.duplicated(["codigoestacion", "timestamp"]).any()
atributos.to_parquet(RUTA_ATR, index=False)
print(f"Guardado: {RUTA_ATR.name} | {len(atributos):,} pares | columnas: {list(atributos.columns)}")

Guardado: negativos_atributos.parquet | 3,973,266 pares | columnas: ['codigoestacion', 'timestamp', 'n_est_pos', 'dist_pos_km', 'n_est_activas', 'horas_a_positivo', 'int_max_ventana', 'razon']


### Conclusión de la Sección 2

El recálculo reproduce exactamente el `n_est_pos` de 11 cuando se incluye a Nueva Generación. Sin ella dejan de ser near-miss 338 negativos de train, 1,945 de val y 1,946 de test: unos 31 instantes por período de validación y prueba en que NG era la única estación positiva de la red. El criterio `red_activa`, que cuenta también las estaciones con lluvia en curso o tormenta inminente, agrega 39,731 negativos de train a los 55,911 del criterio espacial y eleva la bolsa difícil de 13.2 a 22.5 negativos por positivo. El criterio de intensidad suma menos de mil sobre cualquiera de los dos. El criterio temporal (±24 h) amplía la bolsa a 38 por positivo, pero mide cercanía en el tiempo y no semejanza de la escena. Se adopta `red_activa ∪ casi_positivo` como bolsa difícil por defecto del cargador (96,333 negativos de train con L_max ≥ 2); `red` y `temporal` quedan como variantes para S7. Los atributos de cada par quedaron en `negativos_atributos.parquet`.

## 3. Negativos con nube fría (`nube_fria`)

Los criterios anteriores definen la dificultad por lo que ocurre en la red de estaciones. Este la define por lo que ve el satélite: un negativo es difícil si su parche en t tiene una fracción fría comparable a la de los positivos. El resumen es el percentil 10 de C13 sobre el parche de 32 × 32 píxeles, la variable más discriminante del EDA (AUC 0.756 frente a aleatorios con hora emparejada, `12-dataset_eda` Sección 5) y la que usa la referencia de umbral de S5. Son justamente los negativos que esa referencia marcaría como alerta, de modo que el criterio apunta al atajo que la red debe superar: "hay nubes frías cerca".

El umbral se fija con los positivos de entrenamiento (nunca con val o test). Se comparan tres candidatos, los cuartiles del p10 de los positivos de train: un negativo es `nube_fria` si su p10 es igual o más frío que el candidato.

**3.1 Cálculo.** El percentil se calcula para las 63 estaciones en todos los instantes del caché, sobre los enteros empaquetados, y se convierte a K con la escala de la banda (la conversión es afín, así que el resultado es el mismo que calcularlo en K). Se verifica contra el cálculo directo en K sobre una muestra.

In [12]:
from cache_imagenes import Cache, decodificar
from negativos import percentil_banda

RUTA_CACHE = RUTA_BASE / "data" / "cache"
zs    = pd.read_csv(RUTA_BASE / "data_ingestion" / "satellite" / "normalizacion_zscore.csv")
cache = Cache(RUTA_CACHE, zs)
K13   = cache.bandas.index("CMI_C13")

p10 = (percentil_banda(cache, "CMI_C13", 10, progreso=lambda r: tqdm(r, desc="p10 de C13"))
       .rename(columns={"valor": "p10_c13"}))
print(f"Instantes × estaciones: {len(p10):,} | tz: {p10['timestamp'].dt.tz}")

# Verificación contra el percentil calculado en K (como en 12) sobre una muestra
rng = np.random.default_rng(42)
m = p10.iloc[rng.choice(len(p10), 2000, replace=False)]
dif = []
for r in m.itertuples():
    fila = int(cache.fila_de[r.timestamp.tz_convert(None)])
    k_ = decodificar(cache.parche_crudo(fila, int(r.codigoestacion)), cache.scale, cache.offset)[K13]
    dif.append(abs(np.percentile(k_, 10) - r.p10_c13))
print(f"Diferencia máxima con el cálculo en K (2,000 pares): {max(dif):.2e} K")
assert max(dif) < 1e-3

p10 de C13: 100%|██████████| 210/210 [01:32<00:00,  2.27it/s]


Instantes × estaciones: 6,598,935 | tz: UTC
Diferencia máxima con el cálculo en K (2,000 pares): 3.05e-05 K


In [13]:
ind = ind.drop(columns="p10_c13", errors="ignore").merge(p10, on=["codigoestacion", "timestamp"], how="left")
assert ind["p10_c13"].notna().all(), "Pares del índice sin imagen en t en el caché"

pos_tr = ind[(ind["periodo"] == "train") & (ind["etiqueta"] == 1)]
neg_tr = ind[(ind["periodo"] == "train") & (ind["etiqueta"] == 0)]
grupos = {"positivos": pos_tr,
          "difíciles (red_activa ∪ casi)": neg_tr[(neg_tr["n_est_activas"] >= 1) | (neg_tr["banda_int"] == "50–80 %")],
          "resto de negativos": neg_tr[(neg_tr["n_est_activas"] == 0) & (neg_tr["banda_int"] != "50–80 %")]}
print("p10 de C13 en train (K):")
print(pd.DataFrame({g: d["p10_c13"].quantile([.1, .25, .5, .75, .9]) for g, d in grupos.items()}).T.round(1).to_string())

p10 de C13 en train (K):
                                0.10   0.25   0.50   0.75   0.90
positivos                      217.0  228.2  246.6  260.0  267.5
difíciles (red_activa ∪ casi)  217.5  225.8  241.1  258.2  268.0
resto de negativos             237.7  255.6  269.2  275.3  278.7


### 3.2 Umbral del criterio y tamaño de la bolsa

Para cada candidato se reporta cuántos negativos quedan como `nube_fria`, cuántos de ellos son nuevos respecto de la bolsa por defecto (`red_activa ∪ casi_positivo`) y cuántos difíciles por positivo resultan al sumar el criterio. Un umbral muy cálido convierte en "difícil" cualquier día nublado; uno muy frío deja solo convección profunda.

In [14]:
umbrales_p10 = {f"p{q} de positivos": pos_tr["p10_c13"].quantile(q / 100) for q in (25, 50, 75)}
base_dif = (ind["n_est_activas"] >= 1) | (ind["banda_int"] == "50–80 %")

filas = []
for p in ORDEN:
    L = 2 if p == "train" else 4
    msk = (ind["periodo"] == p) & (ind["L_max"] >= L)
    n_pos = int((msk & (ind["etiqueta"] == 1)).sum())
    neg_p = msk & (ind["etiqueta"] == 0)
    for nombre, u in umbrales_p10.items():
        fria = ind["p10_c13"] <= u
        filas.append({"periodo": p, "umbral": f"{nombre} ({u:.1f} K)",
                      "nube_fria": int((neg_p & fria).sum()),
                      "nuevos": int((neg_p & fria & ~base_dif).sum()),
                      "difíciles con nube_fria": int((neg_p & (fria | base_dif)).sum()),
                      "por positivo": round((neg_p & (fria | base_dif)).sum() / n_pos, 1),
                      "% de negativos": f"{(neg_p & (fria | base_dif)).sum() / neg_p.sum():.1%}"})
print(f"Bolsa por defecto en train (red_activa ∪ casi_positivo): {int((ind['periodo'].eq('train') & ind['etiqueta'].eq(0) & base_dif).sum()):,}\n")
print(pd.DataFrame(filas).to_string(index=False))

Bolsa por defecto en train (red_activa ∪ casi_positivo): 96,333

periodo                     umbral  nube_fria  nuevos  difíciles con nube_fria  por positivo % de negativos
  train p25 de positivos (228.2 K)      95908   67764                   164097          38.6          13.2%
  train p50 de positivos (246.6 K)     235773  179964                   276297          65.0          22.3%
  train p75 de positivos (260.0 K)     431227  357013                   453346         106.7          36.6%
    val p25 de positivos (228.2 K)      17969    9619                    60212          40.5          18.6%
    val p50 de positivos (246.6 K)      47133   28093                    78686          52.9          24.3%
    val p75 de positivos (260.0 K)      81492   53184                   103777          69.8          32.0%
   test p25 de positivos (228.2 K)     175183   92603                   451468          36.6          18.9%
   test p50 de positivos (246.6 K)     513876  345438                  

In [15]:
# 3.3 Se agrega p10_c13 a la tabla de atributos del cargador (el umbral se fija como parámetro del cargador)
atributos = ind[["codigoestacion", "timestamp", "n_est_pos", "dist_pos_km", "n_est_activas",
                 "horas_a_positivo", "int_max_ventana", "razon", "p10_c13"]].copy()
assert not atributos.duplicated(["codigoestacion", "timestamp"]).any()
atributos.to_parquet(RUTA_ATR, index=False)
print(f"Guardado: {RUTA_ATR.name} | {len(atributos):,} pares | columnas: {list(atributos.columns)}")

Guardado: negativos_atributos.parquet | 3,973,266 pares | columnas: ['codigoestacion', 'timestamp', 'n_est_pos', 'dist_pos_km', 'n_est_activas', 'horas_a_positivo', 'int_max_ventana', 'razon', 'p10_c13']


### Conclusión de la Sección 3

El percentil 10 de C13 sobre el parche se calculó para los 6,598,935 pares instante–estación del caché y coincide con el cálculo en K (diferencia máxima 3·10⁻⁵ K). En entrenamiento, la bolsa difícil por defecto (`red_activa ∪ casi_positivo`) tiene una distribución de p10 prácticamente igual a la de los positivos (medianas de 241.1 y 246.6 K), mientras que el resto de los negativos es mucho más cálido (269.2 K). El criterio basado en la lluvia observada ya reúne, por tanto, escenas frías comparables a las de los positivos. El criterio `nube_fria` agrega negativos con nubosidad fría sin lluvia fuerte en ninguna estación de la red. Con el umbral en el primer cuartil del p10 de los positivos de train (228.2 K), suma 67,764 negativos nuevos y lleva la bolsa a 38.6 por positivo; con la mediana o el tercer cuartil la bolsa crece a 65 y 107 por positivo (hasta 37 % de los negativos), y el criterio deja de seleccionar escenas difíciles. Se fija `nube_fria` con umbral 228.2 K como variante para S7, frente a la bolsa por defecto. El p10 de cada par quedó en `negativos_atributos.parquet`.

## Anexo A. Exclusión de Nueva Generación del modelado

Nueva Generación (`0021206600`) pasó el control de calidad de `03-ground_truth_qc` y tiene etiquetas en 05–12, con un umbral de excepción (su propio P95 diagnóstico, 16.2 mm/h, el más alto de la red). En la revisión de datos previa al modelado se decidió excluirla de train, val y test sin modificar las notebooks 03–12. Este anexo deja la evidencia en el repositorio.

Los filtros de 03 comparan el P95 diario y la cobertura de días con dato. Una estación cuyo balancín deja de registrar la lluvia débil pasa ambos: sigue transmitiendo y sigue registrando los días de lluvia intensa. Por eso aquí se compara otra cantidad: **la fracción de registros húmedos** (intensidad rolling 60 min ≥ 0.2 mm/h) por temporada, frente a sus vecinas del mismo cluster geográfico (cluster 3) y frente a la red completa. Si el sensor registra con normalidad, esa fracción debe seguir a la de sus vecinas, que reciben la misma lluvia. Se muestra además el número de registros por temporada, para distinguir un sensor que no detecta lluvia de una estación que no transmite.

In [11]:
RUTA_HOR = RUTA_GT / "df_horaria.parquet"                                    # 04 (hora local sin zona)
horaria  = pd.read_parquet(RUTA_HOR)
horaria["codigoestacion"] = cod10(horaria["codigoestacion"])
meta_qc  = pd.read_csv(RUTA_GT / "resumen_qc_66_estaciones_original.csv", dtype={"codigo": str}).set_index("codigo")
print(f"df_horaria: {len(horaria):,} registros | {horaria['codigoestacion'].nunique()} estaciones | "
      f"tz = {horaria['fechaobservacion'].dt.tz}")

TEMP = {12: "DEF", 1: "DEF", 2: "DEF", 3: "MAM", 4: "MAM", 5: "MAM",
        6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
TEMPS = ["DEF", "MAM", "JJA", "SON"]
cluster_ng = meta_qc.loc[NG, "cluster_geografico"]

h = horaria.assign(anio=horaria["fechaobservacion"].dt.year,
                   temporada=horaria["fechaobservacion"].dt.month.map(TEMP),
                   humedo=horaria["intensidad_mm_h"] >= 0.2,
                   cluster=horaria["codigoestacion"].map(meta_qc["cluster_geografico"]))
frac = h.groupby(["anio", "codigoestacion", "temporada"])["humedo"].mean().unstack().reindex(columns=TEMPS) * 100
nreg = h.groupby(["anio", "codigoestacion", "temporada"]).size().unstack().reindex(columns=TEMPS)

tablas_ng = {}
for anio in (2024, 2025):
    f, n = frac.loc[anio], nreg.loc[anio]
    vecinas = [c for c in f.index if meta_qc.loc[c, "cluster_geografico"] == cluster_ng and c != NG]
    t = pd.DataFrame({
        "NG (% húmedos)":          f.loc[NG],
        "vecinas mín":             f.loc[vecinas].min(),
        "vecinas máx":             f.loc[vecinas].max(),
        "red p10":                 f.quantile(0.1),
        "red mediana":             f.quantile(0.5),
        "NG (registros)":          n.loc[NG],
        "vecinas mediana (registros)": n.loc[vecinas].median(),
    }).T
    tablas_ng[anio] = t
    print(f"\n=== {anio} — cluster {cluster_ng}, {len(vecinas)} vecinas ===")
    print(t.round(2).to_string())

df_horaria: 22,774,536 registros | 63 estaciones | tz = None

=== 2024 — cluster 3, 11 vecinas ===
temporada                         DEF       MAM      JJA       SON
NG (% húmedos)                   4.67       NaN      NaN      5.28
vecinas mín                      2.59      2.37     0.17      7.69
vecinas máx                      3.45      3.40     1.21     10.76
red p10                          1.69      1.81     0.44      6.11
red mediana                      2.68      2.58     0.93      8.14
NG (registros)                4324.00       NaN      NaN   5647.00
vecinas mediana (registros)  50867.00  22629.00  8197.00  59475.00

=== 2025 — cluster 3, 11 vecinas ===
temporada                         DEF       MAM       JJA        SON
NG (% húmedos)                   1.39      1.33      0.32       0.07
vecinas mín                      1.45      6.63      3.63       5.31
vecinas máx                      3.52     13.03      7.41       6.39
red p10                          1.52      8.22    

In [12]:
# Positivos aislados: instantes en que NG es la única estación positiva de la red
TZ = "America/Bogota"
utiles_pos = etiquetas[es_pos]
n_pos_t = utiles_pos.groupby("timestamp").size()
ng_pos  = utiles_pos[utiles_pos["codigoestacion"] == NG]
aislados = ng_pos[ng_pos["timestamp"].map(n_pos_t) == 1]
per = lambda ts: pd.Series(np.select([ts.dt.tz_convert(TZ) < pd.Timestamp("2024-11-15", tz=TZ),
                                      ts.dt.tz_convert(TZ) < pd.Timestamp("2025-01-01", tz=TZ)],
                                     ["train", "val"], "test"), index=ts.index)
print(pd.DataFrame({"positivos de NG": per(ng_pos["timestamp"]).value_counts(),
                    "solo NG positiva en la red": per(aislados["timestamp"]).value_counts()})
        .reindex(ORDEN).fillna(0).astype(int).to_string())
print(f"\nUmbral de NG: {umbrales.loc[NG, 'umbral_mm_h']} mm/h | red: mediana {umbrales['umbral_mm_h'].median():.1f}, "
      f"máximo de las demás {umbrales.drop(NG)['umbral_mm_h'].max():.1f} mm/h")

       positivos de NG  solo NG positiva en la red
train                6                           6
val                 55                          42
test                47                          38

Umbral de NG: 16.2 mm/h | red: mediana 6.0, máximo de las demás 12.9 mm/h


In [13]:
# Cifras citadas en el texto del anexo (salidas del 8-oct-2026)
t24, t25 = tablas_ng[2024].round(2), tablas_ng[2025].round(2)
assert t24.loc["NG (% húmedos)", ["DEF", "SON"]].tolist() == [4.67, 5.28]
assert t24.loc[["vecinas mín", "vecinas máx"], "SON"].tolist() == [7.69, 10.76]
assert t25.loc["NG (% húmedos)"].tolist() == [1.39, 1.33, 0.32, 0.07]
assert t25.loc["vecinas mín"].tolist() == [1.45, 6.63, 3.63, 5.31]
assert t25.loc["vecinas máx"].tolist() == [3.52, 13.03, 7.41, 6.39]
print("Cifras del anexo verificadas")

Cifras del anexo verificadas


**Resultado.** En 2024 la fracción de registros húmedos de Nueva Generación es comparable a la de sus vecinas en DEF, pero en SON queda por debajo de todas ellas (5.28 % frente a 7.69–10.76 %) y del percentil 10 de la red. En 2025 la diferencia se agrava en cada temporada: 1.33 % en MAM frente a 6.63–13.03 % de sus vecinas, 0.32 % en JJA frente a 3.63–7.41 % y 0.07 % en SON frente a 5.31–6.39 %, es decir, prácticamente ningún registro húmedo en la segunda temporada de lluvias. Durante esos mismos meses la estación transmite con normalidad: su número de registros por temporada es del mismo orden que el de sus vecinas. Es la firma de un pluviómetro de balancín obstruido, que deja de registrar la lluvia mientras sigue enviando datos.

Esto tiene tres consecuencias. Primero, sus etiquetas de 2025 contienen falsos negativos: sus pares entran como "sin lluvia fuerte" mientras las estaciones vecinas registran lluvia normal. Segundo, su umbral de excepción (16.2 mm/h, el más alto de la red) es probablemente un artefacto instrumental: un sensor que pierde la lluvia débil deja en su distribución sobre todo los eventos intensos, y eso infla el P95. Tercero, el deterioro ya era visible en SON 2024, de modo que tampoco su registro de entrenamiento es confiable. La estación produce además positivos aislados, en instantes en que ninguna otra estación de la red supera su umbral (tabla anterior); estos son coherentes con un registro intermitente, aunque no constituyen por sí solos prueba de falla.

**Decisión.** Nueva Generación se excluye de los tres conjuntos del modelado y no cuenta como estación positiva o activa al definir los negativos difíciles (Secciones 2 y 3). La exclusión se aplica en el cargador y no en 03–12, que quedan como registro del conjunto de datos tal como se construyó. Aporta 56,659 pares del índice (1.4 %) y alrededor de 100 positivos (0.6 %).

## Conclusión

Esta notebook calcula los atributos que deciden a qué bolsa de negativos pertenece cada par del índice, sin modificar las etiquetas de 05 ni el índice de 11. La intensidad en la ventana reproduce la etiqueta sin excepción (Sección 1). Los criterios espaciales se recalcularon sin Nueva Generación, cuya exclusión se documenta en el Anexo A (Sección 2). La bolsa difícil por defecto del cargador es `red_activa ∪ casi_positivo`: 96,333 negativos de entrenamiento (22.7 por positivo), con escenas tan frías como las de los positivos. La variante para S7 añade `nube_fria` con umbral de 228.2 K (38.6 por positivo) (Sección 3). Todos los atributos quedaron en `negativos_atributos.parquet`, que lee el cargador de `15-cargador`.